In [1]:
!pip3 install rdflib morph_kgc pandas sqlalchemy sqlalchemy_dremio


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
from pathlib import Path
import morph_kgc
from urllib.parse import quote_plus

def load_env(path="../.env"):
    env_path = Path(path)
    if not env_path.exists():
        return
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

load_env()

missing = [key for key in ["DREMIO_USER", "DREMIO_PASSWORD"] if key not in os.environ]
if missing:
    raise RuntimeError(f"Missing environment variables: {', '.join(missing)}. Check ../.env or export them before starting Jupyter.")

dremio_user = os.environ["DREMIO_USER"]
dremio_password = os.environ["DREMIO_PASSWORD"]
dremio_host = os.environ.get("DREMIO_HOST", "localhost")
dremio_flight_port = os.environ.get("DREMIO_FLIGHT_PORT", "32010")
db_url = (
    "dremio+flight://"
    f"{quote_plus(dremio_user)}:{quote_plus(dremio_password)}"
    f"@{dremio_host}:{dremio_flight_port}/dremio?UseEncryption=false"
)

config = f"""
[DataSource1]
mappings: ../mapping/mapping.ttl
db_url: {db_url}
"""

g = morph_kgc.materialize(config)

g.serialize("../output/output.ttl", format="turtle")

print("RDF generated!")

INFO | 2026-04-28 22:39:55,790 | Parallelization is not supported for darwin when running as a library. If you need to speed up your data integration pipeline, please run through the command line.
INFO | 2026-04-28 22:39:57,756 | 26 mapping rules retrieved.
INFO | 2026-04-28 22:39:57,809 | Mapping partition with 26 groups generated.
INFO | 2026-04-28 22:39:57,815 | Maximum number of rules within mapping group: 1.
INFO | 2026-04-28 22:39:57,817 | Mappings processed in 1.718 seconds.
INFO | 2026-04-28 22:40:28,563 | Number of triples generated in total: 5181.


RDF generated!
